In [3]:
import pandas as pd
df_results = pd.read_excel('final_result_filtered.xlsx')
df_gt = pd.read_excel('human_ground_truth_new.xlsx') 

In [9]:
import pandas as pd

# 1. Define the 13 canonical fallacy categories
# Standardized as: lowercase, no extra spaces, underscores replaced with spaces
canonical_fallacies = [
    'ad hominem', 'ad populum', 'appeal to emotion', 'circular reasoning', 
    'equivocation', 'fallacy of credibility', 'fallacy of extension', 
    'fallacy of logic', 'fallacy of relevance', 'false causality', 
    'false dilemma', 'faulty generalization', 'intentional'
]

# 2. Define a cleaning function (for reuse)
def clean_label(text):
    if pd.isna(text):
        return None
    # Convert to string -> replace underscores with spaces -> lowercase -> strip whitespace
    return str(text).replace('_', ' ').lower().strip()

# 3. Clean df_gt (Ground Truth)
# Assume ground_truth column is a comma-separated string
def clean_gt_list(gt_str):
    if pd.isna(gt_str):
        return set()
    return {clean_label(item) for item in str(gt_str).split(',')}

df_gt['cleaned_gt_set'] = df_gt['ground_truth'].apply(clean_gt_list)

# 4. Clean df_results (model outputs)
# --- 4. Clean fallacy_option 1, 2, 3 (keep original structure) ---
option_cols = ['fallacy_option1', 'fallacy_option2', 'fallacy_option3']
for col in option_cols:
    df_results[col] = df_results[col].apply(clean_label)

# --- 5. Build the "exploded prediction table" and incorporate no_fallacy logic ---

# A. Identify questions with no answers (all options 1, 2, 3 are null)
# Determined based on model_name and question_id
df_results['is_no_answer'] = df_results[option_cols].isnull().all(axis=1)

# B. Process answered options (retain original logic)
opt1 = df_results[~df_results['is_no_answer']][['model_name', 'question_id', 'fallacy_option1', 'verification_status1']].rename(
    columns={'fallacy_option1': 'prediction', 'verification_status1': 'status'})
opt2 = df_results[~df_results['is_no_answer']][['model_name', 'question_id', 'fallacy_option2', 'verification_status2']].rename(
    columns={'fallacy_option2': 'prediction', 'verification_status2': 'status'})
opt3 = df_results[~df_results['is_no_answer']][['model_name', 'question_id', 'fallacy_option3', 'verification_status3']].rename(
    columns={'fallacy_option3': 'prediction', 'verification_status3': 'status'})

# Concatenate answered rows and remove NaN predictions
# (e.g., if only two options are answered, the third NaN will be removed)
df_answered = pd.concat([opt1, opt2, opt3], ignore_index=True).dropna(subset=['prediction'])

# C. Handle questions with no answers (assign a default no_fallacy entry)
df_no_fallacy = df_results[df_results['is_no_answer']][['model_name', 'question_id']].copy()
df_no_fallacy['prediction'] = 'no_fallacy'
df_no_fallacy['status'] = 'no_pass'  # default status for unanswered cases

# D. Combine both to form the global prediction table (full denominator)
df_preds_global = pd.concat([df_answered, df_no_fallacy], ignore_index=True)

# --- 6. Merge cleaned Ground Truth into the prediction table (unchanged logic) ---
df_preds_global = pd.merge(
    df_preds_global, 
    df_gt[['question_id', 'cleaned_gt_set']], 
    on='question_id', 
    how='left'
)

print("Global dataset preparation completed (including no_fallacy logic)!")
print(f"Total number of predictions (denominator): {len(df_preds_global)}")

# Verify that unanswered questions are also represented as rows
print(f"Number of no_fallacy cases: {len(df_preds_global[df_preds_global['prediction'] == 'no_fallacy'])}")

Global dataset preparation completed (including no_fallacy logic)!
Total number of predictions (denominator): 3621
Number of no_fallacy cases: 90


In [10]:
# Detect cases where questions are labeled as no_fallacy
import pandas as pd
import numpy as np

# 1. Define option-related columns to check
# Based on DataFrame structure, from 'fallacy_option1' to 'explanation3'
option_columns = [
    'fallacy_option1', 'verification_status1', 'fallacy_option2', 'verification_status2',
    'fallacy_option3', 'verification_status3'
]

# 2. Filter rows that match "Empty Case 1"

# Condition a: all option-related columns are empty (NaN / None)
# .isna() checks for empty values, .all(axis=1) ensures all columns are empty
condition_A = df_results[option_columns].isna().all(axis=1)
df_empty_case1 = df_results[condition_A].copy()


# 3. Group results by model_name

# Count how many Empty Case 1 questions per model
empty_case1_counts = (
    df_empty_case1
    .groupby('model_name')
    .size()
    .reset_index(name='Empty_Case1_Count')
)

# Get question_id list for each model
empty_case1_qids = (
    df_empty_case1
    .groupby('model_name')['question_id']
    .apply(list)
    .reset_index(name='Empty_Case1_QIDs')
)

# Merge results
df_diagnostic = pd.merge(
    empty_case1_counts,
    empty_case1_qids,
    on='model_name',
    how='outer'
)

# Format output so models without this case show 0
df_diagnostic = df_diagnostic.fillna(0)
df_diagnostic['Empty_Case1_Count'] = df_diagnostic['Empty_Case1_Count'].astype(int)


# 4. Print report
print("--- Model Diagnostic Report: Empty Case 1 (Context/GT present, but all options empty) ---")
print(df_diagnostic)

--- Model Diagnostic Report: Empty Case 1 (Context/GT present, but all options empty) ---
                            model_name  Empty_Case1_Count  \
0          anthropic/claude-sonnet-4.5                  6   
1          deepseek/deepseek-chat-v3.1                 11   
2                 deepseek/deepseek-r1                 12   
3                     gemini-flash-non                  8   
4                gemini-flash-thinking                  1   
5                       gemini-pro-non                  1   
6                  gemini-pro-thinking                  2   
7          moonshotai/kimi-k2-thinking                 10   
8             openai/gpt-4o-2024-11-20                  6   
9                         openai/gpt-5                  4   
10                 openai/gpt-oss-120b                  7   
11  qwen/qwen3-235b-a22b-thinking-2507                  7   
12           qwen_qwen3-235b-a22b-2507                  3   
13           x-ai/grok-4-fast-thinking                 1

In [7]:
#  Define constants
PASS_STATUS = 'lean_pass'

# Logical failure (new core error type)
LOGICAL_FAIL_STATUS = 'lean_pass_with_type_error'

# Technical failure (independent metric)
TECHNICAL_FAIL_STATUS = 'no_pass'


# Total rows per model (denominator)
total_rows = (
    df_results.groupby('model_name')
    .size()
    .reset_index(name='Total_Count')
)

In [11]:
# --- Cell: Calculate Valid-Correct ---
# Logic: A prediction is considered valid-correct if:
# (1) the predicted label exists in the Ground Truth set of that question, and
# (2) the Lean4 verification status is 'lean_pass'

df_preds_global['is_vc_row'] = df_preds_global.apply(
    lambda row: 1 if (row['prediction'] in row['cleaned_gt_set'] and row['status'] == 'lean_pass') else 0,
    axis=1
)

In [12]:
# --- Cell: Calculate Valid-Alternative ---
# Logic: A prediction is considered valid-alternative if:
# (1) it is NOT in the Ground Truth set,
# (2) it passes Lean4 verification ('lean_pass'), and
# (3) it belongs to one of the 13 predefined canonical fallacies

df_preds_global['is_va_row'] = df_preds_global.apply(
    lambda row: 1 if (row['prediction'] not in row['cleaned_gt_set'] and 
                      row['status'] == 'lean_pass' and 
                      row['prediction'] in canonical_fallacies) else 0,
    axis=1
)

In [13]:
# --- Cell: Calculate Invalid-Correct ---
# Logic: A prediction is considered invalid-correct if:
# (1) the predicted label exists in the Ground Truth set, and
# (2) it fails Lean4 verification (status != 'lean_pass')

df_preds_global['is_ic_row'] = df_preds_global.apply(
    lambda row: 1 if (row['prediction'] in row['cleaned_gt_set'] and row['status'] != 'lean_pass') else 0,
    axis=1
)

In [14]:
# --- Cell: Calculate VF (Verification Failure) ---
# Logic: A prediction is considered verification failure if:
# (1) it is NOT in the Ground Truth set, and
# (2) it results in a logical/type error during Lean4 verification 
#     (e.g., 'lean_pass_with_type_error')

df_preds_global['is_vf_row'] = df_preds_global.apply(
    lambda row: 1 if (row['prediction'] not in row['cleaned_gt_set'] and 
                      row['status'] == 'lean_pass_with_type_error') else 0,
    axis=1
)

# --- Cell: Calculate SF (Syntax Failure) ---
# Logic: A prediction is considered syntax failure if:
# (1) a prediction is provided (i.e., not 'no_fallacy'), and
# (2) the verification status is 'no_pass' (indicating technical/syntax failure)

df_preds_global['is_sf_row'] = df_preds_global.apply(
    lambda row: 1 if (row['prediction'] != 'no_fallacy' and 
                      row['status'] == 'no_pass') else 0,
    axis=1
)

# --- Cell: Calculate NF (No Fallacy) ---
# Logic: The model explicitly outputs no answer ('no_fallacy')

df_preds_global['is_nf_row'] = (df_preds_global['prediction'] == 'no_fallacy').astype(int)

In [15]:
# --- Priority order: VC > VA > IC > (II: VF > SF > NF) ---

# 2. Aggregate by model and question to obtain the classification features for each question
# Using max means that if any option satisfies the condition, the value is set to 1
df_q_logic = df_preds_global.groupby(['model_name', 'question_id']).agg(
    has_vc=('is_vc_row', 'max'),
    has_va=('is_va_row', 'max'),
    has_ic=('is_ic_row', 'max'),
    has_vf=('is_vf_row', 'max'),
    has_sf=('is_sf_row', 'max'),
    has_nf=('is_nf_row', 'max')
).reset_index()

# 3. Perform global mutually exclusive classification
# Assign a single label according to the predefined priority order
def classify_question(row):
    if row['has_vc'] == 1: return 'VC'
    if row['has_va'] == 1: return 'VA'
    if row['has_ic'] == 1: return 'IC'
    if row['has_vf'] == 1: return 'VF'
    if row['has_sf'] == 1: return 'SF'
    return 'NF'

df_q_logic['final_category'] = df_q_logic.apply(classify_question, axis=1)

# 4. Count the distribution of categories for each model across the 107 questions
final_report = df_q_logic.groupby(['model_name', 'final_category']).size().unstack(fill_value=0).reset_index()

# Ensure that all category columns exist
for cat in ['VC', 'VA', 'IC', 'VF', 'SF', 'NF']:
    if cat not in final_report.columns:
        final_report[cat] = 0

# 5. [Additional step] Aggregate Category II (II = VF + SF + NF)
final_report['II'] = final_report['VF'] + final_report['SF'] + final_report['NF']

# 6. Calculate all percentages (fixed denominator = 107)
all_cats = ['VC', 'VA', 'IC', 'II', 'VF', 'SF', 'NF']
for cat in all_cats:
    final_report[f'{cat}_%'] = (final_report[cat] / 107 * 100).round(2)

# 7. Format the output
# Column order: model name, VC%, VA%, IC%, II% (overall failure), VF%, SF%, NF%
display_cols = ['model_name', 'VC_%', 'VA_%', 'IC_%', 'II_%', 'VF_%', 'SF_%', 'NF_%']
print("--- Global performance statistics over 107 questions (including aggregated II and detailed breakdown) ---")
print(final_report[display_cols])

# 8. Validate checksum (VC% + VA% + IC% + II% should equal 100%)
final_report['Checksum_%'] = final_report[['VC_%', 'VA_%', 'IC_%', 'II_%']].sum(axis=1)
print("\nPerformance validation (Checksum_% should be close to 100):")
print(final_report[['model_name', 'Checksum_%']])

# Export the report
final_report[display_cols].to_excel('Verification_Matrix_result.xlsx', index=False)

--- Global performance statistics over 107 questions (including aggregated II and detailed breakdown) ---
final_category                          model_name   VC_%   VA_%  IC_%   II_%  \
0                      anthropic/claude-sonnet-4.5  44.86  47.66  0.93   6.54   
1                      deepseek/deepseek-chat-v3.1  34.58  53.27  1.87  10.28   
2                             deepseek/deepseek-r1  36.45  52.34  0.00  11.21   
3                                 gemini-flash-non  38.32  53.27  0.00   8.41   
4                            gemini-flash-thinking  43.93  50.47  1.87   3.74   
5                                   gemini-pro-non  43.93  53.27  0.93   1.87   
6                              gemini-pro-thinking  46.73  51.40  0.00   1.87   
7                      moonshotai/kimi-k2-thinking  45.79  44.86  0.00   9.35   
8                         openai/gpt-4o-2024-11-20  47.66  46.73  0.00   5.61   
9                                     openai/gpt-5  45.79  50.47  0.00   3.74   
10 